#  Implementación del Cubo de Datos SECOP (Hive)

**Taller ETL – Cubo SECOP**
**Autor:** Jurani Zabala Hernandez


**Objetivo:** Traducir a un modelo tabular en Hive el diseño dimensional definido en `CuboContratosPostgreSQL.drawio` y dejarlo implementado en el *data lake*, listo para ser poblado por `Transformacion.ipynb` y `Cargue.ipynb`. Basado también en el esquema estrella con **7 dimensiones** — incluyendo `dim_tiempo` como **dimensión de rol**, referenciada 3 veces desde el hecho (fecha de firma, inicio y fin) — y **1 tabla de hechos** (`hecho_contratos`).


## Modelo dimensional (de acuerdo al diagrama de cubo propuesto)

**Tabla de hechos: `hecho_contratos`**

| Columna | Tipo | Descripción |
|---|---|---|
| id_contrato | STRING (PK) | Identificador único del contrato |
| sk_entidad | STRING (FK) | - `dim_entidad` |
| sk_proveedor | STRING (FK) | - `dim_proveedor` |
| sk_tiempo_firma | STRING (FK) | - `dim_tiempo` (rol: fecha de firma) |
| sk_tiempo_inicio | STRING (FK) | - `dim_tiempo` (rol: fecha de inicio) |
| sk_tiempo_fin | STRING (FK) | - `dim_tiempo` (rol: fecha de fin) |
| sk_modalidad | STRING (FK) | - `dim_modalidad` |
| sk_ubicacion | STRING (FK) | - `dim_ubicacion` |
| sk_estado | STRING (FK) | - `dim_estado_contrato` |
| sk_categoria | STRING (FK) | - `dim_categoria` |
| valor_contrato, valor_facturado, valor_pagado, valor_pendiente_pago | DOUBLE | Métricas monetarias |
| dias_adicionados | INT | Métrica |
| cantidad_contratos | INT | Métrica aditiva (1 por fila; útil para `SUM` al agregar) |

**Dimensiones**

- `dim_entidad`: sk_entidad, codigo_entidad, nit_entidad, nombre_entidad, orden, sector, rama, centralizada
- `dim_proveedor`: sk_proveedor, codigo_proveedor, tipo_documento, documento_proveedor, nombre_proveedor, es_grupo, es_pyme
- `dim_tiempo`: sk_tiempo, fecha, anio, semestre, trimestre, mes, nombre_mes, dia — **dimensión de rol** (una sola tabla física, referenciada 3 veces desde el hecho).
- `dim_modalidad`: sk_modalidad, modalidad_contratacion, justificacion_modalidad, tipo_contrato, condiciones_entrega
- `dim_ubicacion`: sk_ubicacion, departamento, ciudad, localizacion
- `dim_estado_contrato`: sk_estado, estado_contrato, liquidacion, reversion, habilita_pago_adelantado
- `dim_categoria`: sk_categoria, codigo_categoria_principal, descripcion_proceso, objeto_contrato

## 1. Sesión de Spark con soporte Hive

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SECOP_CuboDatos")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Sesión Spark con soporte Hive inicializada:", spark.version)

26/08/24 01:43:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Sesión Spark con soporte Hive inicializada: 3.1.2


## 2. Creación del esquema (base de datos) del cubo

In [2]:
DB_NAME = "secop_dw"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
spark.catalog.setCurrentDatabase(DB_NAME)
print(f"✅ Base de datos '{DB_NAME}' creada/seleccionada.")
spark.sql("SHOW DATABASES").show(truncate=False)

26/08/24 01:43:15 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/24 01:43:15 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/24 01:43:24 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/08/24 01:43:24 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.6
26/08/24 01:43:24 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/08/24 01:43:25 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/08/24 01:43:25 WARN ObjectStore: Failed to get database secop_dw, returning NoSuchObjectException


✅ Base de datos 'secop_dw' creada/seleccionada.
+---------+
|namespace|
+---------+
|default  |
|secop_dw |
+---------+



## 3. Creación de las 7 tablas de dimensión

In [3]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_entidad (
    sk_entidad       STRING,
    codigo_entidad   STRING,
    nit_entidad      STRING,
    nombre_entidad   STRING,
    orden            STRING,
    sector           STRING,
    rama             STRING,
    centralizada     STRING
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_proveedor (
    sk_proveedor         STRING,
    codigo_proveedor     STRING,
    tipo_documento       STRING,
    documento_proveedor  STRING,
    nombre_proveedor     STRING,
    es_grupo             STRING,
    es_pyme              STRING
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_tiempo (
    sk_tiempo    STRING,
    fecha        DATE,
    anio         INT,
    semestre     INT,
    trimestre    INT,
    mes          INT,
    nombre_mes   STRING,
    dia          INT
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_modalidad (
    sk_modalidad             STRING,
    modalidad_contratacion   STRING,
    justificacion_modalidad  STRING,
    tipo_contrato            STRING,
    condiciones_entrega      STRING
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_ubicacion (
    sk_ubicacion   STRING,
    departamento   STRING,
    ciudad         STRING,
    localizacion   STRING
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_estado_contrato (
    sk_estado                STRING,
    estado_contrato          STRING,
    liquidacion              STRING,
    reversion                STRING,
    habilita_pago_adelantado STRING
)
STORED AS PARQUET
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.dim_categoria (
    sk_categoria              STRING,
    codigo_categoria_principal STRING,
    descripcion_proceso       STRING,
    objeto_contrato           STRING
)
STORED AS PARQUET
""")

print("✅ 7 tablas de dimensión creadas.")

26/08/24 01:44:52 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/08/24 01:44:52 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/08/24 01:44:52 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/24 01:44:52 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/24 01:44:52 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-warehouse/secop_dw.db/dim_entidad specified for non-external table:dim_entidad
26/08/24 01:44:53 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-warehouse/secop_dw.db/dim_proveedor specified for non-external table:dim_proveedor
26/08/24 01:44:53 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-warehouse/secop_dw.db/dim_tiempo specified for non-external table:dim_tiempo
26/08/24 01:44:53 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-

✅ 7 tablas de dimensión creadas.


26/08/24 01:44:54 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-warehouse/secop_dw.db/dim_categoria specified for non-external table:dim_categoria


## 4. Creación de la tabla de hechos

`dim_tiempo` se referencia **3 veces** (`sk_tiempo_firma`, `sk_tiempo_inicio`, `sk_tiempo_fin`) — es una dimensión de rol. Se particiona por `anio_firma` (derivado de la fecha de firma) como optimización física, adicional al diagrama lógico pero estándar en implementaciones Hive de este tipo de cubo.


In [4]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.hecho_contratos (
    id_contrato            STRING,
    sk_entidad             STRING,
    sk_proveedor           STRING,
    sk_tiempo_firma        STRING,
    sk_tiempo_inicio       STRING,
    sk_tiempo_fin          STRING,
    sk_modalidad           STRING,
    sk_ubicacion           STRING,
    sk_estado              STRING,
    sk_categoria           STRING,
    valor_contrato         DOUBLE,
    valor_facturado        DOUBLE,
    valor_pagado           DOUBLE,
    valor_pendiente_pago   DOUBLE,
    dias_adicionados       INT,
    cantidad_contratos     INT
)
PARTITIONED BY (anio_firma INT)
STORED AS PARQUET
""")

print("✅ Tabla de hechos 'hecho_contratos' creada (particionada por anio_firma).")

26/08/24 01:46:09 WARN HiveMetaStore: Location: file:/home/jovyan/work/spark-warehouse/secop_dw.db/hecho_contratos specified for non-external table:hecho_contratos


✅ Tabla de hechos 'hecho_contratos' creada (particionada por anio_firma).


## 5. Evidencia de la estructura del cubo

In [5]:
tablas_cubo = [
    "hecho_contratos", "dim_entidad", "dim_proveedor", "dim_tiempo",
    "dim_modalidad", "dim_ubicacion", "dim_estado_contrato", "dim_categoria",
]

for tabla in tablas_cubo:
    print(f"\n=== Estructura de {DB_NAME}.{tabla} ===")
    spark.sql(f"DESCRIBE {DB_NAME}.{tabla}").show(truncate=False)

print(f"\nTablas creadas en '{DB_NAME}':")
spark.sql(f"SHOW TABLES IN {DB_NAME}").show(truncate=False)


=== Estructura de secop_dw.hecho_contratos ===
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|id_contrato            |string   |null   |
|sk_entidad             |string   |null   |
|sk_proveedor           |string   |null   |
|sk_tiempo_firma        |string   |null   |
|sk_tiempo_inicio       |string   |null   |
|sk_tiempo_fin          |string   |null   |
|sk_modalidad           |string   |null   |
|sk_ubicacion           |string   |null   |
|sk_estado              |string   |null   |
|sk_categoria           |string   |null   |
|valor_contrato         |double   |null   |
|valor_facturado        |double   |null   |
|valor_pagado           |double   |null   |
|valor_pendiente_pago   |double   |null   |
|dias_adicionados       |int      |null   |
|cantidad_contratos     |int      |null   |
|anio_firma             |int      |null   |
|# Partition Information|         |       |
|# col_name             |dat

## Conclusiones

- Se implementó fielmente el esquema estrellapropuesto en el taller: 7 dimensiones + 1 hecho.
- `dim_tiempo` se modeló como **dimensión de rol** (una sola tabla física, referenciada 3 veces desde el hecho), técnica estándar de modelado dimensional para evitar duplicar dimensiones idénticas.
- El cubo queda listo para ser poblado por `Transformacion.ipynb` y `Cargue.ipynb`.